# Which DMSO plate does a stim map use, and does one plate only reduce the noise?

**Question (user, 2026-10-06).** "When we work on a stim, which DMSO do we use, and what is the consequence? I want
to use only DMSO1 for everything — from training to test, for all methods — and know whether that reduces the noise."

Every (donor, stim) of the atlas was measured in three wells, one per plate, each next to its own plate control:
`DMSO1`, `DMSO2`, `DMSO3` (column `drug` of `obs` in `ptb_concatenated_batchcorrected/<celltype>_HVPV.h5ad`).
Unstim has the same three wells, so a plate-matched Unstim → stim map is possible.
Part 1 states what the code does today and measures how much plate-to-plate difference that lets in. Part 2 is the
experiment: the same held-out design (5 repeats × 4 test donors, k = 15 training donors) with five data arms.

## Answer (2026-10-07)

**Short answer: no. Using DMSO1 only (or any single plate) does not reduce the noise, and it makes held-out
prediction worse.** Pooling the three plates, as every pipeline does today, is the better choice. The only place where
one plate "helps" is when the thing to predict is that same plate's well under PI, and there the gain is the plate's
own batch offset, not the donor.

**Which DMSO is used for a stim: all three, pooled, everywhere.** The paper's stim maps, the learning-curve caches
and every held-out score take Unstim *and* stim cells from the DMSO1 + DMSO2 + DMSO3 wells together (about
36 / 33 / 30 % of the cells), with no plate label kept and no plate matching between source and target. Only the drug
arm uses a single control (its plate's `use_DMSO`).

**Consequence.** The three wells of one donor and stim differ by more than sampling:
- **PI**: plate-to-plate SD of the response 0.34 against a between-donor SD of 0.21 (variance ratio 2.6). It is
  plate 1: its PI response is 0.35 larger along the response direction, and it is the odd plate out in 69 % of cases.
  In the pooled truth this plate noise is about as large as the whole between-donor variance (ratio 1.0).
- **Other stims**: plate-only variance is 0.23–0.26 × the between-donor variance (IFNa, IL246, LPS), 0.47–0.53
  (TNFa, GMCSF), 0.76 (IL33). After averaging three plates it is still 8–26 % of the between-donor variance of the
  pooled truth. DMSO1 is only mildly odd outside PI (odd plate out in 35–39 % of cases against 33 % by chance).
- The split-half noise ceiling (0.87) cannot see any of this, because both halves hold the same mix of plates. So
  part of what `where_cellot_skill.ipynb` calls "real donor-to-donor differences" is plate noise. well_ceiling
  estimates a realistic ceiling of ~0.73 on the pooled target.
- Plate matching helps somewhat: Unstim and stim from the same plate share part of the plate offset. The plate-only
  variance of the matched response is 0.60–0.86 × that of the stim level alone (GMCSF, IFNa, IL246, LPS, TNFa),
  but 0.96 for PI, where plate 1 differs in its *response*, not in its baseline.

**Part 2: one plate everywhere (5 repeats × 4 held-out donors, k = 15, CellOT and four closed-form maps).**

| CellOT, held-out donors | pool | poolN1 (pooled, DMSO1 cell count) | DMSO1 | DMSO2 | DMSO3 |
|---|---|---|---|---|---|
| median cells per stim block | 2498 | 948 | 953 | 820 | 712 |
| skill on its own truth | **0.611** | 0.562 | 0.567 | 0.529 | 0.489 |
| skill on the donor's pooled response | **0.612** | 0.600 | 0.442 | 0.553 | 0.489 |
| … PI only | 0.717 | 0.722 | 0.432 | 0.631 | 0.561 |
| … the other six stims | 0.571 | 0.557 | 0.443 | 0.528 | 0.465 |
| better than pool, donor-repeats (pooled response) | — | 0/20 | 0/20 | 0/20 | 0/20 |

1. **Against the donor's real response (the pooled three wells), every single-plate pipeline is worse than the pooled
   one, in 20 of 20 donor-repeats, for every method** (CellOT 0.61 → 0.44 with DMSO1; gauss_ot 0.60 → 0.43;
   quantile_map 0.54 → 0.32). DMSO1 is the worst of the three plates, and not only because of PI: on the other six
   stims it is also last (0.44 vs 0.53 / 0.47).
2. **It is the single plate that costs, not the fewer cells.** Cutting the pooled data to DMSO1's cell count (poolN1,
   2.6× fewer cells) lowers the skill by only 0.01 (0.612 → 0.600; nothing at all for PI). Taking those cells from
   one plate instead lowers it by 0.06–0.17.
3. **On its own one-well truth, DMSO1 reports 0.567, below the pooled pipeline's 0.611 (pool better in 17 of 20)
   and the same as an equally small pooled sample (poolN1 0.562; DMSO1 better in 11 of 20, p 0.62).** One well is
   not a cleaner measurement: its split-half noise is the same as poolN1's (≈ 0.03 arcsinh), larger than the
   pooled block's (≈ 0.02), and two plates of the same donor differ by another 0.045–0.072 on top of that outside PI
   (0.22–0.31 under PI when plate 1 is involved). Averaging three plates removes part of that; one plate keeps all of it.
4. **Plate matching helps only for PI, and only for predicting that plate.** A map trained on plate p predicts the
   donor's plate-p PI well better than the pooled map in 18–20 of 20 donor-repeats for each plate (DMSO1: 0.757 vs
   0.577), because each plate has its own systematic PI offset. For the other stims it is no better (12, 10, 8 of 20;
   p 0.23–0.67). A plain feature-level shift fitted on the same plate does as well as CellOT (0.563 vs 0.567), so what
   is learnt is the plate's batch offset, shared by all donors.
5. **No arm captures more of the real between-donor differences.** CellOT's specific R² is 0.05 (pool), 0.04 (poolN1),
   −0.01 (DMSO1), 0.08 / 0.08 (DMSO2 / DMSO3) on own truth; it varies from −0.16 to +0.19 between repeats, far more
   than between arms. Against the pooled response DMSO1 is clearly negative (−0.12).

**What to do.** Keep pooling the three plates for stim maps and truths. If plate noise matters for a claim, the
replicate reference (§2.3: a donor's other plates, plate-matched) is the honest one-well ceiling (0.55–0.62), and PI
on plate 1 should be flagged as a batch effect (it is the `DMSO1` outlier already seen in the drug arm).

**Caveats.** One k (15). The 20 donor-repeats are not fully independent (14 donors over 5 repeats), so the p-values are
optimistic; the 0/20 and 20/20 results do not depend on that. Skill on own truth is relative to each arm's own effect
size, so arms are not directly comparable there (DMSO1's PI effect is 0.45 vs 0.29 pooled, which inflates its PI
skill); the pooled-response and one-well comparisons are the fair ones. Caches are capped at 10k cells per donor ×
condition; single wells are mostly below the cap, so for large celltypes the pooled arm has ~3× the cells.

## Part 1 — facts

### 1.1 Which wells each pipeline uses (code)

| pipeline | Unstim (source) | stim (target) | plate matched? |
|---|---|---|---|
| paper stim maps `cross_validation_original_13_HVPV` (168 configs, all `drug_used: DMSO`, no `batch_correction`) | all `DMSO*` wells pooled | all `DMSO*` wells pooled | **no** |
| learning curve `lc/cache.py` → `work/lc/cache` (and `cache_full`, `cache_dnorm`) | pooled, ≤ 10k random cells | pooled, ≤ 10k random cells | **no** |
| `heldout/cache_stim_all` (all 196 pairs, same code with `--all-stims`) | pooled | pooled | **no** |
| scoring truth `lc/score.py:truth_tables` (and `where_skill`, `cond_*`) | test donor's pooled cache | test donor's pooled cache | **no** |

- simco `cellot/data/cell.py:81`: with `drug_used: DMSO` the loader keeps `data.obs['drug'].str.startswith("DMSO")`,
  i.e. DMSO1 + DMSO2 + DMSO3. The source/target label comes only from `stim` (`:214`), the split only from
  `split`/`transport` (`:325`, the 80/20 cut grouped by stim, `:394`), so the plate label is dropped. Training
  draws source and target batches independently (`cellot/train/train.py:202-203`): an Unstim cell from plate 1
  can be transported towards stim cells of plate 3. The runtime batch correction (`cell.py:116-119`, which loops over
  the three DMSO treatments) is not used by the paper's stim run (no `batch_correction` key in its configs).
- The drug arm differs: `cell.py:69-77` keeps one control, the plate's `use_DMSO` from `batchcorrection_2.csv`
  (LPZ and SA → DMSO1).
- `lc/cache.py` (same filter, `startswith("DMSO")`, key = (stim, donor), one random permutation per key): the 10k
  cells of a block are a random mix of the three plates in proportion to their cell counts. `lc/cond_train.py`
  pairs source and target by donor, not by plate.
- Test donors: input = her pooled Unstim, truth = her pooled stim quantiles. The split-half noise
  (`lc/score.py:truth_tables`, `x[:h]` vs `x[h:]` of the shuffled block) compares two halves that hold the same mix
  of plates, so it **cannot see a plate effect**.

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_rows", 120)
W = Path("/scratch/users/bmerwan/atlas_heldout/work/lc/dmso")
FIG = W / "fig"; FIG.mkdir(parents=True, exist_ok=True)
FLOOR = 0.0273
F = W / "facts"
S = pd.read_csv(F / "shares.csv")
for x in ["stim", "Unstim"]:
    t = S[[f"{x}_DMSO{i}" for i in (1, 2, 3)]]
    S[[f"{x} share DMSO{i}" for i in (1, 2, 3)]] = t.div(t.sum(1), axis=0).values
print("Share of each plate in the pooled 10k-cell blocks (mean over celltype x donor):")
display(S.groupby("stim")[[c for c in S if "share" in c]].mean().round(3))

### 1.2 How large is the plate effect? (per-well quantiles, ≤ 30k cells per well)

Per (stim, celltype, donor, q10–q90, marker), from well_ceiling's per-well quantiles (`lc/dmso_facts.py`):
- **R** = pooled stim − pooled Unstim (the response the learning curve scores); between-donor SD = SD of R over the 19 donors;
- **r_p** = stim − Unstim on plate p (plate-matched response); its SD across the 3 plates = **plate SD**;
- the same on **pseudo-wells** (the cells re-dealt at random into groups of the well sizes) = sampling part; plate-only = the rest;
- *matched / unmatched plate-only var*: plate variance of r_p (Unstim from the same plate) over that of the stim level
  alone (Unstim from all plates). < 1 means the plate offset is partly shared by Unstim and stim of a plate, so
  matching removes it.
Live features: |mean R| > 0.0273. SDs are root-mean-square over cases (arcsinh units).

In [ ]:
T = pd.read_csv(F / "by_stim.csv", index_col=0)
cols = ["features", "response |R| median", "between-donor SD of R", "plate-only SD of r", "sampling SD of r (pseudo-wells)",
        "plate-only SD of Unstim level", "plate-only var / between-donor var", "plate var in pooled truth / between var",
        "matched / unmatched plate-only var", "median plate-only SD of r", "median between-donor SD of R"]
display(T[cols].round(3))
P = pd.read_csv(F / "by_stim_plate.csv")
print("How far each plate's response sits from the other two (dev_p = r_p - mean of the other two plates):")
display(P.pivot(index="stim", columns="plate", values="RMS systematic (mean over donors)").round(3)
        .add_prefix("systematic RMS ").join(P.pivot(index="stim", columns="plate", values="mean signed dev x sign(R)").round(3)
        .add_prefix("signed (along R) ")).join(P.pivot(index="stim", columns="plate",
        values="share of cases where this plate is the odd one").round(2).add_prefix("odd one out ")))
fig, ax = plt.subplots(figsize=(8, 3.6))
st = [s for s in T.index if s != "all"]; x = np.arange(len(st)); w = 0.27
for i, (c, lab) in enumerate([("between-donor SD of R", "between donors"), ("plate-only SD of r", "between plates (same donor)"),
                              ("sampling SD of r (pseudo-wells)", "sampling (one well)")]):
    ax.bar(x + (i - 1) * w, T.loc[st, c], w, label=lab)
ax.set_xticks(x, st); ax.set_ylabel("SD of the response (arcsinh)"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Plate-to-plate vs donor-to-donor differences in the stim response", fontsize=10); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "plate_vs_donor.png", dpi=150)

**Reading 1.** Shown as SDs, the plate effect is the same size as the sampling noise of one well for most stims
(0.07–0.10), and below the between-donor SD (0.13–0.19). PI is the exception: its plate SD (0.34) is larger than
everything else, and it is driven by plate 1. The systematic part (the same offset for every donor) is 0.54 for
DMSO1 under PI, and 0.03–0.07 for every other plate × stim. So plate 1 under PI is a batch shift shared by all
donors. Elsewhere, plate differences are mostly donor-specific well noise. Caveats: features with tiny cell counts
inflate the RMS values, so the medians in the table are smaller. The between-donor SD itself contains some plate and
sampling noise.

## Part 2 — one plate only, everywhere (held-out donors)

Five arms, each its own cache (Unstim and stim from the same wells, ≤ 10k cells per donor × condition), each used
for training, for the test donor's input and for her truth:

| arm | wells | why |
|---|---|---|
| `pool` | DMSO1+2+3 (today's cache) | reference |
| `poolN1` | pooled, cut to the DMSO1 cell count of each block | same cells count as DMSO1, plates still mixed: isolates "fewer cells" |
| `DMSO1` | plate 1 only, Unstim → stim plate-matched | **the requested arm** (plate 1 = the PI outlier plate) |
| `DMSO2`, `DMSO3` | plate 2 / 3 only | controls: plate 1 specifically, or plate matching in general? |

Methods: CellOT (`lc/cond_train --ctx none`, paired by donor, 20k iterations, retrained for every arm including
`pool`), gauss_ot, quantile_map, median_shift (`lc/baselines`), feature-level shift; identity. k = 15. Scored by
`lc/dmso_score.py` → `work/lc/dmso/cases.pkl`. Tests are on the 20 donor-repeats, never on cases.

Three readouts, because each arm's own truth is a different measurement:
1. **own truth** — each arm scored against its own wells (what a DMSO1-only pipeline would report);
2. **common pooled truth** — every arm's prediction scored against the test donor's pooled three-well response (the
   best estimate of her real response): does training/inputting one plate predict the donor better?
3. **common DMSO1 truth** — the target is one well; adds the *replicate* reference (her own response on plate 2/3,
   plate-matched, transferred to plate 1), i.e. the best any model can do against one well.

In [ ]:
import os; CF = Path(os.environ.get("DMSO_CASES", W / "cases.pkl"))   # env override = smoke tests only
READY = CF.exists()
if not READY:
    print("pending: work/lc/dmso/cases.pkl not written yet (jobs in work/lc/STATUS.md)")
else:
    D = pd.read_pickle(CF)
    for c in ["arm", "stim", "celltype", "donor", "marker"]: D[c] = D[c].astype(str)
    METH = ["CellOT", "gauss_ot", "quantile_map", "median_shift", "shift"]
    ARMS = [a for a in ["pool", "poolN1", "DMSO1", "DMSO2", "DMSO3"] if a in set(D.arm)]
    avail = D.groupby("arm")[[m for m in METH if m != "shift"]].apply(lambda x: x.notna().mean()).round(2)
    print("fraction of rows with a prediction:"); display(avail)
    KEY = ["repeat", "stim", "celltype", "donor", "q", "marker"]
    D["effect"] = (D.t - D.u).abs()

### 2.1 Own truth: what a one-plate pipeline would report

skill = 1 − |pred − t| / |t − u| (median per repeat, mean over the 5 repeats), effects > 0.0273 on the arm's own
truth; *ceiling* = split-half ceiling of that truth (blind to plate effects); *n* = cases per repeat.

In [ ]:
def skill_tab(X, pred, t, u, by=None):
    sk = 1 - (X[pred] - X[t]).abs() / (X[t] - X[u]).abs()
    g = [X.repeat] + ([X[b] for b in by] if by else [])
    m = sk.groupby(g).median()
    return m.groupby(level=list(range(1, m.index.nlevels))).mean() if by else m.mean()

if READY:
    rows = {}
    for a in ARMS:
        X = D[(D.arm == a) & (D.effect > FLOOR)]
        r = {m: skill_tab(X.dropna(subset=[m]), m, "t", "u") if X[m].notna().any() else np.nan for m in METH}
        r["ceiling"] = (1 - X.noise / X.effect).groupby(X.repeat).median().mean()
        r["n"] = len(X) / X.repeat.nunique(); r["median cells (stim)"] = X.n_s.median()
        rows[a] = r
    display(pd.DataFrame(rows).T.round(3))
    print("CellOT and gauss_ot by stim (own truth):")
    out = {}
    for a in ARMS:
        X = D[(D.arm == a) & (D.effect > FLOOR)]
        for m in ["CellOT", "gauss_ot"]:
            if X[m].notna().any(): out[(m, a)] = skill_tab(X.dropna(subset=[m]), m, "t", "u", ["stim"])
    display(pd.DataFrame(out).round(3))

In [ ]:
# Paired tests on own truth: per donor-repeat median skill (20 donor-repeats); sanity check against the learning curve.
def dr_skill(X, m, t="t", u="u"):
    s = 1 - (X[m] - X[t]).abs() / (X[t] - X[u]).abs()
    return s.groupby([X.repeat, X.donor]).median()

def paired(A, B, higher_better=True):
    d = (A - B).dropna(); w = (d > 0) if higher_better else (d < 0)
    return {"mean diff": d.mean(), "A better (x/20)": f"{w.sum()}/{len(d)}", "p (Wilcoxon)": wilcoxon(d).pvalue}

if READY:
    Xp = D[(D.arm == "pool") & (D.effect > FLOOR)]
    print(f"sanity: pool arm CellOT (retrained) {skill_tab(Xp, 'CellOT', 't', 'u'):.3f} vs the learning curve's paired "
          f"CellOT none on the same cases {skill_tab(Xp.dropna(subset=['CellOT_lcnone']), 'CellOT_lcnone', 't', 'u'):.3f}")
    OWN = {(a, m): dr_skill(D[(D.arm == a) & (D.effect > FLOOR)], m) for a in ARMS for m in ["CellOT", "gauss_ot", "shift"]}
    rows = []
    for m in ["CellOT", "gauss_ot", "shift"]:
        for A, B in [("poolN1", "pool"), ("DMSO1", "pool"), ("DMSO1", "poolN1"), ("DMSO2", "poolN1"), ("DMSO3", "poolN1"),
                     ("DMSO1", "DMSO2"), ("DMSO1", "DMSO3")]:
            rows.append({"method": m, "A vs B": f"{A} vs {B}", **paired(OWN[(A, m)], OWN[(B, m)])})
    print("own truth, paired skill per donor-repeat (A better = higher skill):")
    display(pd.DataFrame(rows).set_index(["method", "A vs B"]).round(4))
    print("own truth without PI (CellOT, gauss_ot):")
    display(pd.DataFrame({a: {m: skill_tab(D[(D.arm == a) & (D.effect > FLOOR) & (D.stim != "PI")], m, "t", "u")
                              for m in ["CellOT", "gauss_ot"]} for a in ARMS}).T.round(3))

**Reading 2.1.** *Sanity check:* the `pool` arm, retrained here, gives CellOT 0.611 against 0.615 for the learning
curve's paired CellOT on the same cases, and gauss_ot 0.595, quantile_map 0.538, median_shift 0.512, the learning
curve's k = 15 values. So the arm reproduces the reference.

Each arm judged on its own wells: pool 0.611, poolN1 0.562, DMSO1 0.567, DMSO2 0.529, DMSO3 0.489 (CellOT; the same
order for every method). Paired over the 20 donor-repeats:
- fewer cells costs 0.05: poolN1 is below pool in 20 of 20;
- one plate instead of a mixed sample of the same size changes nothing for DMSO1 (better in 11 of 20, p 0.62; gauss_ot
  12 of 20, p 0.23) and costs 0.03–0.07 for DMSO2 / DMSO3 (worse in 15 and 19 of 20). DMSO2/3 wells are smaller
  (820 / 712 cells) and their ceilings lower (0.78 / 0.76 vs DMSO1 0.81);
- DMSO1 is ahead of DMSO2 / DMSO3 (15 and 19 of 20) only because of PI: DMSO1's PI response is larger (0.45 vs
  0.21–0.26), and skill is measured relative to the effect, so a bigger effect gives a bigger skill. DMSO1 is the only
  arm above pool on any stim, and only on PI (0.757 vs 0.717). Without PI: pool 0.571, poolN1 0.510, DMSO1 0.497,
  DMSO2 0.479, DMSO3 0.442.

So a DMSO1-only pipeline would report a lower skill than today's, and the drop is what one would get from simply
having fewer cells. The split-half ceilings (0.87 pool, 0.76–0.81 for the others) follow the cell counts and cannot
see plate effects.

### 2.2 Common target: the donor's pooled three-plate response

Every arm's prediction against the `pool` arm's truth (t0, u0); cases with |t0 − u0| > 0.0273 present in every arm.
*better than pool in x/20* = donor-repeats where the arm's median |error| is below the `pool` arm's (same method);
Wilcoxon over the 20 donor-repeats.

In [ ]:
if READY:
    P0 = D[D.arm == "pool"][KEY + ["t", "u"]].rename(columns={"t": "t0", "u": "u0"})
    C = D.merge(P0, on=KEY)
    C = C[(C.t0 - C.u0).abs() > FLOOR]
    res = []
    for m in METH:
        if C[m].isna().all(): continue
        am = [a for a in ARMS if C.loc[C.arm == a, m].notna().any()]      # arms that have this method
        if "pool" not in am: continue
        Cm = C[C.arm.isin(am)].dropna(subset=[m]); Cm = Cm[Cm.groupby(KEY).arm.transform("nunique") == len(am)]
        e = (Cm[m] - Cm.t0).abs(); dr = e.groupby([Cm.arm, Cm.repeat, Cm.donor]).median().unstack(0)
        for a in am:
            Xa = Cm[Cm.arm == a]
            row = {"method": m, "arm": a, "skill vs pooled truth": skill_tab(Xa, m, "t0", "u0"),
                   "median |err|": e[Cm.arm == a].median()}
            if a != "pool" and a in dr:
                d = dr[a] - dr["pool"]
                row["better than pool (donor-repeats)"] = f"{(d < 0).sum()}/{d.notna().sum()}"
                row["p (Wilcoxon)"] = wilcoxon(d.dropna()).pvalue if d.notna().sum() > 5 else np.nan
            res.append(row)
    R22 = pd.DataFrame(res).set_index(["method", "arm"]); display(R22.round(4))
    print("skill vs pooled truth, by stim:")
    BS = {}
    for m in ["CellOT", "gauss_ot", "quantile_map"]:
        Cm = C.dropna(subset=[m])
        if not len(Cm): continue
        for a in ARMS: BS[(m, a)] = skill_tab(Cm[Cm.arm == a], m, "t0", "u0", ["stim"])
    BS = pd.DataFrame(BS); display(BS.round(3))
    m0 = "CellOT" if ("CellOT", "pool") in BS else "gauss_ot"
    fig, ax = plt.subplots(figsize=(9, 3.6)); st = list(BS.index); x = np.arange(len(st)); w = 0.8 / len(ARMS)
    for i, a in enumerate(ARMS):
        if (m0, a) in BS: ax.bar(x + (i - (len(ARMS) - 1) / 2) * w, BS[(m0, a)], w, label=a)
    ax.set_xticks(x, st); ax.set_ylabel(f"{m0} skill vs the donor's\npooled 3-plate response"); ax.axhline(0, color="k", lw=.6)
    ax.legend(frameon=False, fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    ax.set_title("Held-out donors, k = 15: which plate wells to train on and input (higher = better)", fontsize=10)
    ax.grid(axis="y", alpha=.3)
    fig.tight_layout(); fig.savefig(FIG / "skill_by_stim_pooled_truth.png", dpi=150)

In [ ]:
# CellOT vs pooled truth, PI and the other stims separately; paired median |error| per donor-repeat (A better = lower).
if READY:
    rows = []
    for sub, Cs in [("other stims", C[C.stim != "PI"]), ("PI", C[C.stim == "PI"])]:
        e = (Cs.CellOT - Cs.t0).abs().groupby([Cs.arm, Cs.repeat, Cs.donor]).median().unstack(0)
        for A, B in [("poolN1", "pool"), ("DMSO1", "pool"), ("DMSO2", "pool"), ("DMSO3", "pool"), ("DMSO1", "poolN1"),
                     ("DMSO2", "poolN1"), ("DMSO3", "poolN1"), ("DMSO1", "DMSO2")]:
            rows.append({"stims": sub, "A vs B": f"{A} vs {B}", "skill A": skill_tab(Cs[Cs.arm == A], "CellOT", "t0", "u0"),
                         "skill B": skill_tab(Cs[Cs.arm == B], "CellOT", "t0", "u0"), **paired(e[A], e[B], higher_better=False)})
    display(pd.DataFrame(rows).set_index(["stims", "A vs B"]).round(4))

**Reading 2.2.** This is the main test: same target for everyone, the test donor's pooled three-well response.
- **Every single-plate arm is worse than pool in 20 of 20 donor-repeats, for every method** (CellOT 0.612 → DMSO1
  0.442, DMSO2 0.553, DMSO3 0.489). The method differences (CellOT 0.612, feature shift 0.605, gauss_ot 0.597,
  quantile_map 0.538) are much smaller than the arm differences.
- **Fewer cells is nearly free**: poolN1 0.600 vs 0.612 (better than pool in 2 of 20 on the other stims; on PI 0.722 vs
  0.717, 9 of 20, p 0.7).
- **One plate is not**: against poolN1, which has the same number of cells, DMSO1 is worse in 20 of 20 (PI 0.432 vs
  0.722; other stims 0.443 vs 0.557), DMSO2 in 16–18 of 20, DMSO3 in 18–20 of 20.
- **Plate 1 specifically is the worst**, on PI (0.43 vs 0.63 / 0.56) and on the other stims (0.44 vs 0.53 / 0.47;
  worse than DMSO2 in 18 of 20). IL33 is negative in every single-plate arm (−0.23 for DMSO1): a single well's IL33
  response is mostly plate noise.

Plate matching (Unstim and stim from the same plate) does not compensate for losing the average over three plates.

### 2.3 Common target: one well (DMSO1), and the replicate reference

Truth = the test donor's DMSO1 well (t1, u1). Predictions: each arm's map as is, plus two **replicates** (no model):
her own plate-matched response on another plate moved onto her DMSO1 Unstim, u1 + (t_p − u_p), p = 2, 3, and their
mean. Repeated with DMSO2 as the target so plate 1 is not the only one judged.

In [ ]:
def one_well(target):
    others = [p for p in ["DMSO1", "DMSO2", "DMSO3"] if p != target]
    if not {target, *others} <= set(ARMS): return None
    base = D[D.arm == target][KEY + ["t", "u"]].rename(columns={"t": "tt", "u": "ut"})
    for p in others:
        base = base.merge(D[D.arm == p][KEY + ["t", "u"]].rename(columns={"t": "t_" + p, "u": "u_" + p}), on=KEY)
        base["rep_" + p] = base.ut + base["t_" + p] - base["u_" + p]
    base["rep_mean"] = base.ut + np.mean([base["t_" + p] - base["u_" + p] for p in others], 0)
    base = base[(base.tt - base.ut).abs() > FLOOR]
    out = {}
    for p in others + ["mean"]: out[f"replicate ({p})"] = skill_tab(base, "rep_" + p, "tt", "ut")
    for m in ["CellOT", "gauss_ot", "quantile_map", "shift"]:
        for a in ARMS:
            X = base.merge(D[D.arm == a][KEY + [m]], on=KEY).dropna(subset=[m])
            if len(X): out[f"{m} [{a}]"] = skill_tab(X, m, "tt", "ut")
    rs = {p: skill_tab(base, "rep_" + p, "tt", "ut", ["stim"]) for p in others}
    return pd.Series(out), pd.DataFrame(rs)

if READY:
    for tg in ["DMSO1", "DMSO2"]:
        r = one_well(tg)
        if r is None: print(f"pending: {tg} target needs all three single-plate arms"); continue
        print(f"target = {tg} well:"); display(r[0].round(3).to_frame("skill"))
        print(f"replicate skill by stim (target {tg}):"); display(r[1].round(3))

In [ ]:
# One-well targets, CellOT: the plate-matched arm vs pool / poolN1 / the donor's own replicate wells (mean of the other two
# plates), paired median |error| per donor-repeat (A better = lower), PI and the other stims separately.
if READY and {"DMSO1", "DMSO2", "DMSO3"} <= set(ARMS):
    rows = []
    for tg in ["DMSO1", "DMSO2", "DMSO3"]:
        oth = [p for p in ["DMSO1", "DMSO2", "DMSO3"] if p != tg]
        B = D[D.arm == tg][KEY + ["t", "u"]].rename(columns={"t": "tt", "u": "ut"})
        for p in oth: B = B.merge(D[D.arm == p][KEY + ["t", "u"]].rename(columns={"t": "t_" + p, "u": "u_" + p}), on=KEY)
        B["replicate"] = B.ut + np.mean([B["t_" + p] - B["u_" + p] for p in oth], 0)
        B = B[(B.tt - B.ut).abs() > FLOOR]
        for a in ["pool", "poolN1", tg]: B = B.merge(D[D.arm == a][KEY + ["CellOT"]].rename(columns={"CellOT": a}), on=KEY)
        for sub, Bs in [("all", B), ("other stims", B[B.stim != "PI"]), ("PI", B[B.stim == "PI"])]:
            e = {k: (Bs[k] - Bs.tt).abs().groupby([Bs.repeat, Bs.donor]).median() for k in ["pool", "poolN1", tg, "replicate"]}
            r = {"target": tg, "stims": sub, **{f"skill {'matched' if k == tg else k}": skill_tab(Bs, k, "tt", "ut") for k in ["pool", "poolN1", tg, "replicate"]}}
            for ref in ["pool", "poolN1", "replicate"]:
                pr = paired(e[tg], e[ref], higher_better=False)
                r[f"matched better than {ref}"] = pr["A better (x/20)"]; r[f"p vs {ref}"] = pr["p (Wilcoxon)"]
            rows.append(r)
    display(pd.DataFrame(rows).set_index(["target", "stims"]).round(4))

**Reading 2.3.** Target = one well. The *replicate* (the donor's own plate-matched response measured on the other two
plates) is the reference no model can be expected to beat for donor-specific reasons: 0.565 for a DMSO1 target,
0.622 for DMSO2, 0.551 for DMSO3. Paired, CellOT, plate-matched arm vs the others:

| target well | stims | matched | pool | poolN1 | replicate | matched better than pool | than replicate |
|---|---|---|---|---|---|---|---|
| DMSO1 | PI | 0.757 | 0.577 | 0.568 | 0.445 | 18/20 | 20/20 |
| DMSO1 | other six | 0.498 | 0.510 | 0.494 | 0.604 | 12/20 (p 0.23) | 0/20 |
| DMSO2 | PI | 0.688 | 0.630 | 0.631 | 0.725 | 18/20 | 9/20 |
| DMSO2 | other six | 0.479 | 0.501 | 0.484 | 0.597 | 10/20 (p 0.67) | 0/20 |
| DMSO3 | PI | 0.644 | 0.368 | 0.373 | 0.281 | 20/20 | 20/20 |
| DMSO3 | other six | 0.442 | 0.505 | 0.490 | 0.597 | 8/20 (p 0.37) | 0/20 |

- **Under PI each plate has its own systematic offset**, so a map trained on that plate predicts that plate's well
  better than the pooled map (18–20 of 20 for every plate) and, for plates 1 and 3, better than the donor's own wells
  on the other plates. Here the plate effect is larger than the donor effect. The feature-level shift fitted on DMSO1
  does as well as CellOT (0.563 vs 0.567): the gain is a batch offset shared by all donors, not donor biology.
- **For the other six stims, matching gives nothing** (8–12 of 20, n.s.), and all models stay far below the replicate
  (0.44–0.51 vs 0.60, 0 of 20): there is real donor-specific signal in a well that none of the maps capture. Against an equally small mixed sample (poolN1) the matched map is slightly better at
  predicting its own plate (13–16 of 20; p 0.02 for DMSO1, 0.08 and 0.13 for DMSO2/3), but its skill is not higher
  (0.50 vs 0.49, 0.48 vs 0.48, 0.44 vs 0.49): a small, inconsistent edge.

### 2.4 Noise in each truth: split-half vs plate-to-plate

Per stim, median over live cases: effect |t − u| (own truth), split-half noise of t, and the plate-to-plate
difference of the plate-matched response |(t_p − u_p) − (t_p' − u_p')| between single-plate arms.

In [ ]:
if READY:
    N = D[D.effect > FLOOR].groupby(["stim", "arm"]).agg(effect=("effect", "median"), split_half=("noise", "median"),
                                                          cells=("n_s", "median")).unstack("arm")
    display(N.round(3))
    pl = [p for p in ["DMSO1", "DMSO2", "DMSO3"] if p in ARMS]
    if len(pl) == 3:
        M = D[D.arm.isin(pl)].assign(r=lambda x: x.t - x.u).pivot_table(index=KEY, columns="arm", values="r")
        M = M.join(P0.set_index(KEY)); M = M[(M.t0 - M.u0).abs() > FLOOR]
        PP = pd.DataFrame({f"{a} vs {b}": (M[a] - M[b]).abs().groupby(level="stim").median()
                           for a, b in [("DMSO1", "DMSO2"), ("DMSO1", "DMSO3"), ("DMSO2", "DMSO3")]})
        PP["pooled effect"] = (M.t0 - M.u0).abs().groupby(level="stim").median()
        print("plate-to-plate |difference of plate-matched responses| (held-out test donors, <= 10k cells per well):")
        display(PP.round(3))

### 2.5 Is the donor-specific part better captured? (specific R²)

As in `where_cellot_skill.ipynb`: r = t − shift, rc = r centred over the repeat's test donors; specific R² =
1 − Σ(predicted rc − rc)² / Σ rc². Computed against each arm's own truth and against the pooled truth (shift and
centring from the `pool` arm there).

In [ ]:
GRP = ["repeat", "stim", "celltype", "q", "marker"]
def spec(X, m, t, sh):
    X = X.dropna(subset=[m]).copy()
    X["n"] = X.groupby(GRP)[t].transform("size"); X = X[X.n >= 3]
    r = X[t] - X[sh]; rc = r - r.groupby([X[c] for c in GRP]).transform("mean")
    h = X[m] - X[sh]; hc = h - h.groupby([X[c] for c in GRP]).transform("mean")
    return 1 - ((hc - rc) ** 2).sum() / (rc ** 2).sum()

if READY:
    S0 = D[D.arm == "pool"][KEY + ["shift"]].rename(columns={"shift": "shift0"})
    rows = {}
    for a in ARMS:
        X = D[(D.arm == a) & (D.effect > FLOOR)]
        Y = C[C.arm == a].merge(S0, on=KEY)
        rows[a] = {f"{m} own": spec(X, m, "t", "shift") for m in ["CellOT", "gauss_ot"] if X[m].notna().any()}
        rows[a].update({f"{m} vs pooled": spec(Y, m, "t0", "shift0") for m in ["CellOT", "gauss_ot"] if Y[m].notna().any()})
    display(pd.DataFrame(rows).T.round(3))
    print("CellOT specific R2 on own truth, per repeat (5 repeats x 4 test donors):")
    display(pd.DataFrame({a: [spec(D[(D.arm == a) & (D.effect > FLOOR) & (D.repeat == r)], "CellOT", "t", "shift")
                              for r in sorted(D.repeat.unique())] for a in ARMS}).round(3).rename_axis("repeat"))

**Reading 2.4–2.5.** *Noise.* The split-half noise of one well (0.027–0.038) is the same as that of a pooled sample of
the same size (poolN1, 0.027–0.035) and larger than the pooled block's (0.017–0.024). A single plate is therefore not
less noisy; it only has fewer cells. On top of sampling, two plates of the same held-out donor differ by 0.045–0.072
in the plate-matched response outside PI (about 1.5–2× one well's sampling noise, and a third to two thirds of the
pooled effect; more than the effect for IL33), and by 0.22–0.31 under PI when plate 1 is one of them (0.089 for
DMSO2 vs DMSO3). The pooled truth averages this; a one-plate truth carries all of it, and the split-half ceiling does
not see it.

*Specific R²* (share of the real between-donor differences, beyond the average response, that CellOT predicts):
0.05 for pool, 0.04 poolN1, −0.01 DMSO1, 0.08 DMSO2, 0.08 DMSO3 on own truth; against the pooled response 0.05,
0.02, −0.12, 0.01, −0.01. Per repeat it ranges from −0.16 to +0.19 within one arm, so the arm differences are not
meaningful, except that DMSO1 is the lowest in 3 of 5 repeats. No choice of plate makes the donor-specific part
predictable.

## Part 3 — can the plate be recognised from the cells? And how big is the shared well effect? (2026-10-07)

**Design fact (user, 2026-10-07): each well holds all donors (barcoded), one (stim or Unstim, plate) per well.** So
whatever is common to a well is common to every donor in it: that is the only plate signature a map trained on other
donors could ever learn. Checked first: the single-plate caches are built per cell from its demultiplexed `drug` label
(`lc/cache.py:68`) and partition the pooled cache exactly (pDCs: 182 + 159 + 129 = 470 cells, 100 % found in the pool);
each arm trains, feeds the test donor and scores her on its own plate's cells (`meta.json` `cache_dir`).

`lc/plate_classify.py`: per (condition, celltype), PI excluded, Unstim included, 13 functional markers, **held-out
donors** (GroupKFold over donors), chance 1/3:
- *cell*: one cell → plate (gradient boosting);
- *bag*: a batch of 100 cells (5 quantiles × 13 markers) → plate (logistic regression);
- *bag_dc*: the same, each donor's batches centred on her own mean batch (no plate label used): "given a donor's
  batches, which one is plate k?".

`lc/plate_magnitude.py`: per donor, the 5 × 13 quantiles of each plate's well, centred on her 3-plate mean. The
**shared well offset** = the part of that deviation common to all donors (learnable); **donor × well** = the rest
(new for every donor and well, not learnable); both for the level (one well) and for the response (stim well −
Unstim well of the same plate, i.e. what a plate-matched map sees). Compared with the between-donor SD (of the
3-plate means) and the response size.

In [ ]:
CL = W / "classify"
A = pd.read_csv(CL / "accuracy.csv")
print("accuracy on held-out donors (chance 0.33): median over celltypes")
display(A.pivot_table(index="condition", columns="readout", values="accuracy", aggfunc="median").round(3))
print("bag_dc: recall per plate (median over celltypes)")
display(A[A.readout == "bag_dc"].groupby("condition")[["recall_DMSO1", "recall_DMSO2", "recall_DMSO3"]].median().round(3))
M = pd.read_csv(CL / "plate_magnitude.csv")
print("plate effect size (arcsinh, median over celltypes)")
display(M.groupby(["kind", "condition"])[["shared_well_rms", "donor_x_well_rms", "between_donor_sd", "response_size"]]
        .median().round(3))
R = M[(M.kind == "response") & (M.condition != "PI")].copy()
R["true donor SD (noise removed)"] = np.sqrt((R.between_donor_sd ** 2 - R.donor_x_well_rms ** 2 / 3).clip(lower=0))
print("response, PI excluded: between-donor SD of the 3-plate mean vs the donor x well noise it still contains")
display(R.groupby("condition")[["between_donor_sd", "donor_x_well_rms", "true donor SD (noise removed)"]].median().round(3))

### 3b. The same decomposition on one footing (variance components, all SDs)

Supersedes the magnitude table above, which mixed statistics (an uncorrected RMS of plate offsets, a corrected SD, a
mean of SDs and a mean absolute response). `lc/plate_variance.py`: for one feature, donor d, plate p,
x[d, p] = mu + a[d] + w[p] + e[d, p] (every well holds all donors). SD_donor = real differences between donors,
SD_well = well effect shared by all donors (learnable), SD_dxw = donor × well + cell sampling (not learnable); each
estimated from the two-way table with the noise contributions removed, as an SD per feature, then RMS over the 65
features, median over celltypes. *mean_response* = RMS of the average response, for scale. Last columns: the share of a
donor's apparent deviation that is real, for one well (a[d] + e) and for the 3-well average (a[d] + e / 3).

In [ ]:
V = pd.read_csv(CL / "plate_variance.csv")
T = V.groupby(["kind", "condition"])[["mean_response", "SD_donor", "SD_well", "SD_dxw"]].median()
T["real share, one well"] = T.SD_donor ** 2 / (T.SD_donor ** 2 + T.SD_dxw ** 2)
T["real share, 3 wells"] = T.SD_donor ** 2 / (T.SD_donor ** 2 + T.SD_dxw ** 2 / 3)
display(T.round(3))

**How to read Part 3.**
- **First table:** how often the plate is guessed right on donors the classifier never saw (chance 0.33). It does this
  from one cell, from a batch of 100 cells, or from a batch compared with the same donor's other batches. In that last
  case each batch is centred on the donor's mean batch, which removes her own level.
- **Second table:** the recall for each plate in that last setting.
- **3b:** the size of each component, all as standard deviations in arcsinh units:
  - *SD_donor*: real differences between donors;
  - *SD_well*: the well effect shared by every donor in the well, which is learnable;
  - *SD_dxw*: donor × well, including cell sampling, which is not learnable.
  The *level* rows are one well's values; the *response* rows are the stim well minus the Unstim well of the same plate.
  The first magnitude table is superseded by 3b (2026-10-07: it mixed an uncorrected RMS, a corrected SD, a mean of SDs
  and a mean absolute value, so its columns could not be compared).

**Reading.**
- **One cell does not tell its plate** (0.37–0.40), and neither does a lone batch of 100 cells (0.44–0.56).
- **Batches of the same donor do:** 0.75–0.85, Unstim included, best for plate 1 (recall 0.86–0.95). So every well
  carries a consistent signature shared by all its donors.
- **But it is small in the response.**
  - Outside PI the shared well effect is SD 0.035–0.05 on one well. In the response (stim − Unstim on the same plate) it
    is 0.026–0.036, and only 0.011–0.013 for IL33 / TNFa.
  - That is all a plate-matched map can learn, against average responses of 0.38–0.68 (GMCSF, IFNa, IL246, LPS). This
    is why matching gained nothing outside PI. Under PI the shared well effect is 0.28, as large as the real donor
    differences.
- **Donor × well is 2–4× the shared part:** SD 0.054–0.084 in the response. It cannot be learned, and pooling three
  wells divides its variance by 3.
- **How much of a donor's apparent response is real** (response rows):
  - IFNa / IL246 / LPS: about 80 % from one well, about 92 % from the 3-well average (real donor SD 0.11–0.13, against
    donor × well 0.054–0.064).
  - GMCSF / TNFa: about 54 % from one well, about 78 % from three.
  - IL33: 21 % from one well, 44 % from three. Most of an IL33 donor's apparent individuality is well noise.
- **Implication:**
  - On the 3-well averages the targets are mostly real for the strong stims, so the noise does not explain most of the
    plateau there. The unexplained donor-specific response is real but not predictable from Unstim (where_cellot_skill).
  - For TNFa, IL33 and one-well targets, well noise sets a low ceiling.
  - Split-half noise misses all of it.